# 03 — Feature engineering

The available decision inputs are Time, Amount, and V1–V28. Class is the target. The PCA components have no published meanings, so feature combinations cannot be given customer or merchant interpretations. The split is frozen and all rows remain in the population.

Notebook 02 already applies `log1p(Amount)` and standardization inside its logistic pipeline; these fitted operations use training rows only. Trees in notebook 04 take raw numeric inputs because their threshold splits do not require scaling. This stage tests one additional hypothesis: a smooth 24-hour phase of elapsed Time might help a linear classifier capture a repeating pattern. Its phase is relative to the first transaction, **not** a known local clock hour.

In [1]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
env = os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MLFLOW_DISABLE_AGENT_HINT="1")
run = subprocess.run([sys.executable, str(root / "scripts/run_feature_engineering.py")], cwd=root, env=env, capture_output=True, text=True)
print(run.stdout)
if run.returncode:
    print(run.stderr)
run.check_returncode()

               name  average_precision  brier_score  fraud_canceled_recall  legitimate_cancellation_rate  simulated_profit_per_1000
logistic_regression           0.741775     0.000810               0.657895                      0.000305               21811.863741
logistic_time_cycle           0.741458     0.000817               0.671053                      0.000282               21814.813638
Selection only; calibration and evaluation splits were not scored.



In [2]:
report = json.loads((root / "reports/feature-engineering-results.json").read_text())
assert report["metadata"]["evaluated_split"] == "selection"
assert not report["metadata"]["evaluation_split_scored"]
metrics = pd.DataFrame([report["baseline_logistic"], report["time_cycle_logistic"]])
display(metrics[["name", "rows", "frauds", "average_precision", "brier_score", "fraud_canceled_recall", "legitimate_cancellation_rate", "simulated_profit_per_1000"]].round(6))

,name,rows,frauds,average_precision,brier_score,fraud_canceled_recall,legitimate_cancellation_rate,simulated_profit_per_1000
0,logistic_regression,42632,76,0.741775,0.000810,0.657895,0.000305,21811.863741
1,logistic_time_cycle,42632,76,0.741458,0.000817,0.671053,0.000282,21814.813638


The phase feature barely changes the result: average precision is slightly lower and illustrative profit is about 2.95 source units higher per 1,000 payments. Two days cannot establish a stable daily cycle, and the dataset has no local clock origin. **Decision:** retain the original 30 inputs for notebook 04. The engineered phase remains an exploratory comparison, not a selected model input. Calibration and evaluation are untouched.